# 09 · 3D segmenter to ONNX: check on real scans

1. Clone the repo and install the package (+ MONAI, ONNX).
2. Run `python -m neurolens_ml.export_seg --cases 6`: export `segmenter3d.onnx`, compare one window with PyTorch, then run 6 real val patients through the **torch-free CPU path** (`infer_seg`: NIfTI → RAS → crop → normalize → ONNX sliding window) and compare its masks with PyTorch on the GPU, its Dice with the expert masks, and the measured side, lobe and volume.
3. Show the slices with left/right and front/back marked, to check side and lobe by eye.

**Kaggle setup:** GPU T4 x2, Internet on, inputs `dschettler8845/brats-2021-task1` and the output of the `07_segmentation` notebook. Run with *Save Version → Save & Run All*, then download everything from *Output*.

In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sanjardevx/neurolens.git"
BRANCH = "develop"
CONFIG = "ml/configs/seg_segresnet.yaml"
CASES = 6
REPO_DIR = Path(os.environ.get("NEUROLENS_REPO_DIR", "/tmp/neurolens"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
EXTRA_ARGS = os.environ.get("NEUROLENS_ARGS", "").split()  # only for local smoke tests (fake data, checkpoint)

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
pkgs = {"neurolens_ml": None, "monai": "monai", "nibabel": "nibabel", "onnx": "onnx", "onnxruntime": "onnxruntime",
        "onnxscript": "onnxscript"}
missing = [m for m in pkgs if importlib.util.find_spec(m) is None]
if "neurolens_ml" in missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_DIR / "ml")], check=True)
if "monai" in missing:  # --no-deps: keep Kaggle's own torch
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], check=True)
if rest := [pkgs[m] for m in missing if m not in ("neurolens_ml", "monai")]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *rest], check=True)
print("repo commit:", commit)

In [ ]:
cmd = [sys.executable, "-m", "neurolens_ml.export_seg", "--config", str(REPO_DIR / CONFIG), "--cases", str(CASES),
       "--out", str(OUT_DIR), *EXTRA_ARGS]
print(" ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
assert proc.wait() == 0, "export check failed, see the log above"

In [ ]:
import pandas as pd
from IPython.display import Image, display

report = json.loads((OUT_DIR / "onnx_seg_report.json").read_text())
report["repo_commit"] = commit
(OUT_DIR / "onnx_seg_report.json").write_text(json.dumps(report, indent=2))
display(Image(filename=str(OUT_DIR / "onnx_seg_cases.png")))
pd.read_csv(OUT_DIR / "onnx_seg_cases.csv")